In [ ]:
!pip install jams mir_eval autochord tf_keras onnxruntime
!pip install basic-pitch==0.4.0 --no-deps
!pip install pretty_midi resampy

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # Force CPU — avoids CUDA errors with Basic Pitch

import jams
import mir_eval
import librosa
import numpy as np
import pandas as pd
import autochord
from pathlib import Path

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Paths
GUITARSET_DIR = Path('/content/drive/MyDrive/Capstone/FullGuitarSetData')
AUDIO_DIR = GUITARSET_DIR / 'AudioFiles'
ANNOTATIONS_DIR = GUITARSET_DIR / 'JamsFiles'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.3/350.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 41.5 MB/s eta 0:00:00
  Created wheel for vamp: filename=vamp-1.1.0-cp312-cp312-linux_x86_64.whl size=1705502 sha256=8959b25664b717479fc28e8341d8e64a8a2db4045cbb1c9aa307fdbe3a3682d8
  Stored in directory: /root/.cache/pip/wheels/5a/62/ea/1580385eea4c45f8e2530c6d9a3e4b144dc8a09fdb41e9cf8b
Successfully built vamp
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.3/758.3 kB 14.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 70.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 19.3 MB/s eta

Downloading...
From: https://drive.google.com/uc?id=1XBn7FyYjF8Ff6EuC7PjwwPzFBLRXGP7n
To: /root/.autochord/model.zip
100%|██████████| 2.26M/2.26M [00:00<00:00, 119MB/s]


autochord: Chord model downloaded in /root/.autochord/chroma-seq-bilstm-crf-v1/
autochord: Loaded model from /root/.autochord/chroma-seq-bilstm-crf-v1/
Mounted at /content/drive


In [ ]:
# ============================================================
# Cell 2: GuitarSet Loader
# ============================================================
# Loads a GuitarSet recording's audio path and parsed ground truth
# annotations into a single structured dict for downstream evaluation.
#
# GuitarSet recordings follow a naming convention like:
#   00_BN1-129-Eb_comp
#   ^   ^   ^   ^  ^
#   |   |   |   |  └─ comp (chord backing) or solo (melodic)
#   |   |   |   └──── key
#   |   |   └──────── tempo in BPM
#   |   └──────────── style (BN, Funk, Jazz, Rock, SS)
#   └──────────────── recording ID

# Standard guitar tuning — MIDI pitch of each open string
# Index 0 = low E (string 6 in guitar notation)
# Index 5 = high E (string 1 in guitar notation)
OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]


def load_guitarset_recording(recording_id, audio_dir=AUDIO_DIR, annotations_dir=ANNOTATIONS_DIR):
    """
    Load a GuitarSet recording's audio path and ground truth annotations.

    Args:
        recording_id: Filename stem without extension, e.g. "00_BN1-129-Eb_comp"
        audio_dir: Path to folder containing .wav files
        annotations_dir: Path to folder containing .jams files

    Returns:
        dict with keys:
            - id: the recording_id string
            - audio_path: full path to the .wav file
            - duration: length of the recording in seconds
            - style: e.g. "BN1", "Funk1", "Jazz2"
            - tempo: BPM as int
            - key_in_filename: key as named in the filename, e.g. "Eb"
            - is_comp: True for comping, False for solo
            - key: ground truth key as labeled in the JAMS file (e.g. "Eb:major")
            - chords: list of (start, end, label) tuples for ground truth chords
            - notes: list of dicts with keys (start, duration, string, fret, midi, note_name)
            - beats: list of beat onset times in seconds
            - jam: the raw jams object (in case you need to dig deeper)

    Raises:
        FileNotFoundError: if either the audio or annotation file is missing.
    """
    audio_path = Path(audio_dir) / f"{recording_id}_mic.wav"
    jams_path = Path(annotations_dir) / f"{recording_id}.jams"

    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")
    if not jams_path.exists():
        raise FileNotFoundError(f"Annotation file not found: {jams_path}")

    # Parse filename for metadata
    # Format: "00_Style-TEMPO-KEY_compORsolo"
    stem = recording_id
    parts = stem.split('_')
    # parts[0] = "00", parts[1] = "BN1-129-Eb", parts[2] = "comp" or "solo"
    style_tempo_key = parts[1].split('-')
    style = style_tempo_key[0]
    tempo = int(style_tempo_key[1])
    key_in_filename = style_tempo_key[2]
    is_comp = (parts[2] == 'comp')

    # Load the JAMS annotation
    jam = jams.load(str(jams_path))

    # --- Extract key (song-level) ---
    key_label = None
    key_anns = jam.search(namespace='key_mode')
    if key_anns and len(key_anns[0].data) > 0:
        key_label = key_anns[0].data[0].value

    # --- Extract chord progression (time-aligned) ---
    chords = []
    chord_anns = jam.search(namespace='chord')
    if chord_anns:
        for obs in chord_anns[0].data:
            chords.append((obs.time, obs.time + obs.duration, obs.value))

    # --- Extract beat onsets ---
    beats = []
    beat_anns = jam.search(namespace='beat_position')
    if beat_anns:
        beats = [obs.time for obs in beat_anns[0].data]

    # --- Extract per-string note annotations and derive fret positions ---
    notes = []
    note_anns = jam.search(namespace='note_midi')
    for string_idx, anno in enumerate(note_anns):
        for obs in anno.data:
            midi_pitch = obs.value
            fret = round(midi_pitch - OPEN_STRING_MIDI[string_idx])
            notes.append({
                'start': obs.time,
                'duration': obs.duration,
                'string': string_idx,            # 0 = low E, 5 = high E
                'fret': fret,                    # 0 = open
                'midi': round(midi_pitch),
                'note_name': _midi_to_note_name(midi_pitch),
            })
    # Sort chronologically
    notes.sort(key=lambda n: n['start'])

    return {
        'id': recording_id,
        'audio_path': str(audio_path),
        'duration': jam.file_metadata.duration,
        'style': style,
        'tempo': tempo,
        'key_in_filename': key_in_filename,
        'is_comp': is_comp,
        'key': key_label,
        'chords': chords,
        'notes': notes,
        'beats': beats,
        'jam': jam,
    }


def _midi_to_note_name(midi):
    """Convert MIDI pitch number to note name string (e.g. 64 -> 'E4')."""
    pitch_classes = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    octave = int(midi) // 12 - 1
    pc = pitch_classes[int(midi) % 12]
    return f"{pc}{octave}"


# Smoke test — make sure it works on the file you already have
test_id = "00_BN1-129-Eb_comp"
try:
    rec = load_guitarset_recording(test_id)
    print(f"Loaded: {rec['id']}")
    print(f"  Style: {rec['style']} | Tempo: {rec['tempo']} | Key (filename): {rec['key_in_filename']}")
    print(f"  Comping: {rec['is_comp']} | Duration: {rec['duration']:.2f}s")
    print(f"  Key (JAMS):   {rec['key']}")
    print(f"  Chords:       {len(rec['chords'])} segments")
    print(f"  Notes:        {len(rec['notes'])} per-string annotations")
    print(f"  Beats:        {len(rec['beats'])} markers")
    print(f"\n  First 3 chord segments:")
    for s, e, lab in rec['chords'][:3]:
        print(f"    {s:5.2f}s → {e:5.2f}s  {lab}")
    print(f"\n  First 3 notes:")
    for n in rec['notes'][:3]:
        print(f"    {n['start']:5.2f}s  string={n['string']} fret={n['fret']} {n['note_name']}")
except FileNotFoundError as e:
    print(f"❌ {e}")
    print(f"\nCheck that the file is at the expected path. Looking in:")
    print(f"  Audio: {AUDIO_DIR}")
    print(f"  Annotations: {ANNOTATIONS_DIR}")

Loaded: 00_BN1-129-Eb_comp
  Style: BN1 | Tempo: 129 | Key (filename): Eb
  Comping: True | Duration: 22.32s
  Key (JAMS):   Eb:major
  Chords:       6 segments
  Notes:        133 per-string annotations
  Beats:        48 markers

  First 3 chord segments:
     0.00s →  7.44s  D#:maj
     7.44s → 11.16s  G#:maj
    11.16s → 14.88s  D#:maj

  First 3 notes:
     0.05s  string=1 fret=6 D#3
     0.05s  string=4 fret=6 F4
     0.05s  string=3 fret=7 D4


In [ ]:
# ============================================================
# Cell 3: Pipeline Wrappers
# ============================================================
# Thin wrappers around our three detection tools.
# Each function takes an audio path and returns predictions
# in a format that matches the corresponding ground truth field
# from load_guitarset_recording().

from basic_pitch.inference import predict as basic_pitch_predict


# ---------- Key Detection ----------

# Krumhansl-Schmuckler key profiles
_MAJOR_PROFILE = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09,
                            2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
_MINOR_PROFILE = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53,
                            2.54, 4.75, 3.98, 2.69, 3.34, 3.17])
_PITCH_CLASSES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']


def run_key_detection(audio_path):
    """
    Detect the key of an audio file using Krumhansl-Schmuckler on chromagram.

    Returns:
        dict with keys:
            - key: predicted key label, e.g. "D# major"
            - confidence_gap: gap between top-1 and top-2 (rough confidence)
            - ranked: top 5 candidates with correlation scores
    """
    y, sr = librosa.load(audio_path, sr=None)
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
    chroma_avg = np.mean(chroma, axis=1)

    correlations = {}
    for tonic in range(12):
        major_template = np.roll(_MAJOR_PROFILE, tonic)
        minor_template = np.roll(_MINOR_PROFILE, tonic)
        correlations[f"{_PITCH_CLASSES[tonic]} major"] = np.corrcoef(chroma_avg, major_template)[0, 1]
        correlations[f"{_PITCH_CLASSES[tonic]} minor"] = np.corrcoef(chroma_avg, minor_template)[0, 1]

    ranked = sorted(correlations.items(), key=lambda x: x[1], reverse=True)
    top_key, top_score = ranked[0]
    _, second_score = ranked[1]

    return {
        'key': top_key,
        'confidence_gap': top_score - second_score,
        'ranked': ranked[:5],
    }


# ---------- Chord Detection ----------

def run_chord_detection(audio_path):
    """
    Detect chord progression using autochord.

    Returns:
        list of (start, end, label) tuples in the same shape as
        ground truth chord annotations from the loader.
    """
    chords = autochord.recognize(audio_path)
    # autochord already returns (start, end, label) tuples
    return [(start, end, label) for start, end, label in chords]


# ---------- Note Detection ----------

def run_note_detection(audio_path):
    """
    Detect notes using Basic Pitch.

    Returns:
        list of dicts with keys (start, duration, midi, note_name).
        Note: no string/fret info — that's our fretboard algorithm's job later.
    """
    _, _, note_events = basic_pitch_predict(audio_path)
    notes = []
    for start, end, pitch_midi, amplitude, _ in note_events:
        # Filter out low-amplitude artifacts and out-of-guitar-range notes
        if amplitude < 0.3 or pitch_midi < 40 or pitch_midi > 88:
            continue
        notes.append({
            'start': start,
            'duration': end - start,
            'midi': int(round(pitch_midi)),
            'note_name': _midi_to_note_name(pitch_midi),
            'amplitude': amplitude,
        })
    notes.sort(key=lambda n: n['start'])
    return notes


# ---------- Smoke Test ----------

test_id = "00_BN1-129-Eb_comp"
rec = load_guitarset_recording(test_id)

print(f"Running pipeline on: {rec['id']}\n")

print("Key detection...")
key_pred = run_key_detection(rec['audio_path'])
print(f"  Predicted: {key_pred['key']} (gap={key_pred['confidence_gap']:.3f})")
print(f"  Truth:     {rec['key']}\n")

print("Chord detection...")
chord_pred = run_chord_detection(rec['audio_path'])
print(f"  Predicted: {len(chord_pred)} segments")
print(f"  Truth:     {len(rec['chords'])} segments\n")

print("Note detection...")
note_pred = run_note_detection(rec['audio_path'])
print(f"  Predicted: {len(note_pred)} notes")
print(f"  Truth:     {len(rec['notes'])} per-string note annotations")

Running pipeline on: 00_BN1-129-Eb_comp

Key detection...
  Predicted: D# major (gap=0.091)
  Truth:     Eb:major

Chord detection...
1/1 [==============================] - 1s 838ms/step
  Predicted: 9 segments
  Truth:     6 segments

Note detection...
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/00_BN1-129-Eb_comp_mic.wav...
  Predicted: 147 notes
  Truth:     133 per-string note annotations


In [ ]:
# ============================================================
# Cell 4: Evaluation Metrics (FIXED)
# ============================================================

import mir_eval
import re


# ---------- Key Evaluation ----------

def _normalize_key_for_mir_eval(key_label):
    """
    mir_eval.key.weighted_score expects keys in format like 'C major' or 'A minor'.
    Handle common input formats: 'Eb:major', 'D# major', 'C:maj', etc.
    """
    if key_label is None:
        return None

    # Replace ':' separator with space: 'Eb:major' -> 'Eb major'
    label = key_label.replace(':', ' ')

    # Normalize whitespace
    label = ' '.join(label.split())

    # Split into tonic and mode
    tokens = label.split()
    if len(tokens) < 2:
        return None

    tonic = tokens[0]
    mode = ' '.join(tokens[1:]).lower()

    # Map common mode variants to mir_eval's expected values
    mode_map = {
        'maj': 'major',
        'major': 'major',
        'min': 'minor',
        'minor': 'minor',
    }
    mode = mode_map.get(mode, mode)

    if mode not in ('major', 'minor'):
        return None

    return f"{tonic} {mode}"


def evaluate_key(predicted_key, true_key):
    """
    Score key prediction against truth using mir_eval weighted scoring.
    Returns dict with 'score', 'exact_match', 'predicted', 'truth'.
    """
    pred_norm = _normalize_key_for_mir_eval(predicted_key)
    truth_norm = _normalize_key_for_mir_eval(true_key)

    if pred_norm is None or truth_norm is None:
        return {'score': 0.0, 'exact_match': False, 'predicted': predicted_key, 'truth': true_key}

    try:
        score = mir_eval.key.weighted_score(truth_norm, pred_norm)
    except Exception as e:
        print(f"  Warning: key eval failed: {e}  (pred='{pred_norm}', truth='{truth_norm}')")
        score = 0.0

    return {
        'score': score,
        'exact_match': score == 1.0,
        'predicted': predicted_key,
        'truth': true_key,
    }


# ---------- Chord Evaluation ----------

def _chords_to_mir_eval_format(chord_list):
    """Convert [(start, end, label), ...] to (intervals_array, labels_list)."""
    if not chord_list:
        return np.array([]).reshape(0, 2), []
    intervals = np.array([[start, end] for start, end, _ in chord_list])
    labels = [label for _, _, label in chord_list]
    return intervals, labels


def evaluate_chords(predicted_chords, true_chords):
    """
    Score chord progression against truth using mir_eval.
    Reports three accuracy levels: root, majmin, triads.
    """
    pred_intervals, pred_labels = _chords_to_mir_eval_format(predicted_chords)
    truth_intervals, truth_labels = _chords_to_mir_eval_format(true_chords)

    if len(pred_intervals) == 0 or len(truth_intervals) == 0:
        return {'root_acc': 0.0, 'majmin_acc': 0.0, 'triads_acc': 0.0}

    # Clip both interval sequences to a common time range so mir_eval is happy.
    # We use the intersection: max start, min end across both.
    t_min = max(truth_intervals[0, 0], pred_intervals[0, 0])
    t_max = min(truth_intervals[-1, 1], pred_intervals[-1, 1])

    truth_intervals, truth_labels = mir_eval.util.adjust_intervals(
        truth_intervals, truth_labels, t_min=t_min, t_max=t_max,
        start_label='N', end_label='N'
    )
    pred_intervals, pred_labels = mir_eval.util.adjust_intervals(
        pred_intervals, pred_labels, t_min=t_min, t_max=t_max,
        start_label='N', end_label='N'
    )

    # Merge so both have the same interval boundaries
    intervals, ref_labels, est_labels = mir_eval.util.merge_labeled_intervals(
        truth_intervals, truth_labels, pred_intervals, pred_labels
    )
    durations = mir_eval.util.intervals_to_durations(intervals)

    root_comparisons   = mir_eval.chord.root(ref_labels, est_labels)
    majmin_comparisons = mir_eval.chord.majmin(ref_labels, est_labels)
    triads_comparisons = mir_eval.chord.triads(ref_labels, est_labels)

    return {
        'root_acc':   mir_eval.chord.weighted_accuracy(root_comparisons, durations),
        'majmin_acc': mir_eval.chord.weighted_accuracy(majmin_comparisons, durations),
        'triads_acc': mir_eval.chord.weighted_accuracy(triads_comparisons, durations),
    }


# ---------- Note Detection Evaluation ----------

def evaluate_notes(predicted_notes, true_notes, onset_tolerance=0.05, pitch_tolerance=0.5):
    """
    Score note detection by matching predicted notes to ground truth notes.
    Returns precision, recall, f1, and raw counts.
    """
    if not predicted_notes or not true_notes:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0,
                'n_pred': len(predicted_notes), 'n_truth': len(true_notes)}

    pred_intervals = np.array([[n['start'], n['start'] + n['duration']] for n in predicted_notes])
    pred_pitches = np.array([n['midi'] for n in predicted_notes], dtype=float)

    truth_intervals = np.array([[n['start'], n['start'] + n['duration']] for n in true_notes])
    truth_pitches = np.array([n['midi'] for n in true_notes], dtype=float)

    # Convert MIDI to Hz for mir_eval
    pred_freqs = 440.0 * (2.0 ** ((pred_pitches - 69) / 12.0))
    truth_freqs = 440.0 * (2.0 ** ((truth_pitches - 69) / 12.0))

    precision, recall, f1, _ = mir_eval.transcription.precision_recall_f1_overlap(
        truth_intervals, truth_freqs,
        pred_intervals, pred_freqs,
        onset_tolerance=onset_tolerance,
        pitch_tolerance=pitch_tolerance,
        offset_ratio=None,
    )

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'n_pred': len(predicted_notes),
        'n_truth': len(true_notes),
    }


# ---------- Smoke Test ----------

print(f"Evaluating predictions on: {rec['id']}\n")

key_eval = evaluate_key(key_pred['key'], rec['key'])
print(f"Key:    predicted='{key_eval['predicted']}'  truth='{key_eval['truth']}'  score={key_eval['score']:.2f}")

chord_eval = evaluate_chords(chord_pred, rec['chords'])
print(f"Chords: root={chord_eval['root_acc']:.2f}  majmin={chord_eval['majmin_acc']:.2f}  triads={chord_eval['triads_acc']:.2f}")

note_eval = evaluate_notes(note_pred, rec['notes'])
print(f"Notes:  precision={note_eval['precision']:.2f}  recall={note_eval['recall']:.2f}  f1={note_eval['f1']:.2f}")
print(f"        ({note_eval['n_pred']} predicted vs {note_eval['n_truth']} truth)")

Evaluating predictions on: 00_BN1-129-Eb_comp

Key:    predicted='D# major'  truth='Eb:major'  score=1.00
Chords: root=0.76  majmin=0.76  triads=0.76
Notes:  precision=0.70  recall=0.77  f1=0.74
        (147 predicted vs 133 truth)


In [ ]:
# ============================================================
# Cell 5: Batch Runner
# ============================================================
# Run the full pipeline + evaluation across multiple GuitarSet
# recordings, returning a DataFrame of per-recording results.

def evaluate_pipeline_on_recording(recording_id):
    """Run the full pipeline + eval on one recording. Returns a flat dict."""
    rec = load_guitarset_recording(recording_id)

    key_pred = run_key_detection(rec['audio_path'])
    chord_pred = run_chord_detection(rec['audio_path'])
    note_pred = run_note_detection(rec['audio_path'])

    key_eval = evaluate_key(key_pred['key'], rec['key'])
    chord_eval = evaluate_chords(chord_pred, rec['chords'])
    note_eval = evaluate_notes(note_pred, rec['notes'])

    return {
        'recording_id': recording_id,
        'style': rec['style'],
        'tempo': rec['tempo'],
        'key_truth': rec['key'],
        'is_comp': rec['is_comp'],
        'duration': rec['duration'],

        # key
        'key_pred': key_pred['key'],
        'key_score': key_eval['score'],
        'key_exact': key_eval['exact_match'],
        'key_gap': key_pred['confidence_gap'],

        # chords
        'chord_root_acc': chord_eval['root_acc'],
        'chord_majmin_acc': chord_eval['majmin_acc'],
        'chord_triads_acc': chord_eval['triads_acc'],

        # notes
        'note_precision': note_eval['precision'],
        'note_recall': note_eval['recall'],
        'note_f1': note_eval['f1'],
        'notes_predicted': note_eval['n_pred'],
        'notes_truth': note_eval['n_truth'],
    }


def run_batch_evaluation(recording_ids, verbose=True):
    """
    Run evaluation across multiple recordings.

    Args:
        recording_ids: list of recording stems like ["00_BN1-129-Eb_comp", ...]
        verbose: whether to print progress

    Returns:
        pandas DataFrame with one row per recording
    """
    results = []
    for i, rid in enumerate(recording_ids):
        if verbose:
            print(f"[{i+1}/{len(recording_ids)}] Processing {rid}...")
        try:
            result = evaluate_pipeline_on_recording(rid)
            results.append(result)
        except FileNotFoundError as e:
            print(f"  ⚠ Skipping: {e}")
        except Exception as e:
            print(f"  ❌ Error: {e}")

    df = pd.DataFrame(results)
    return df


# ---------- Discover what recordings you have ----------

available_recordings = sorted([
    p.stem.replace('_mic', '')
    for p in AUDIO_DIR.glob('*_mic.wav')
])
print(f"Found {len(available_recordings)} recordings:")
for r in available_recordings:
    print(f"  {r}")

Found 360 recordings:
  00_BN1-129-Eb_comp
  00_BN1-129-Eb_solo
  00_BN1-147-Gb_comp
  00_BN1-147-Gb_solo
  00_BN2-131-B_comp
  00_BN2-131-B_solo
  00_BN2-166-Ab_comp
  00_BN2-166-Ab_solo
  00_BN3-119-G_comp
  00_BN3-119-G_solo
  00_BN3-154-E_comp
  00_BN3-154-E_solo
  00_Funk1-114-Ab_comp
  00_Funk1-114-Ab_solo
  00_Funk1-97-C_comp
  00_Funk1-97-C_solo
  00_Funk2-108-Eb_comp
  00_Funk2-108-Eb_solo
  00_Funk2-119-G_comp
  00_Funk2-119-G_solo
  00_Funk3-112-C#_comp
  00_Funk3-112-C#_solo
  00_Funk3-98-A_comp
  00_Funk3-98-A_solo
  00_Jazz1-130-D_comp
  00_Jazz1-130-D_solo
  00_Jazz1-200-B_comp
  00_Jazz1-200-B_solo
  00_Jazz2-110-Bb_comp
  00_Jazz2-110-Bb_solo
  00_Jazz2-187-F#_comp
  00_Jazz2-187-F#_solo
  00_Jazz3-137-Eb_comp
  00_Jazz3-137-Eb_solo
  00_Jazz3-150-C_comp
  00_Jazz3-150-C_solo
  00_Rock1-130-A_comp
  00_Rock1-130-A_solo
  00_Rock1-90-C#_comp
  00_Rock1-90-C#_solo
  00_Rock2-142-D_comp
  00_Rock2-142-D_solo
  00_Rock2-85-F_comp
  00_Rock2-85-F_solo
  00_Rock3-117-Bb_comp

In [ ]:
# ============================================================
# Cell 6: Run Evaluation and Analyze Results
# ============================================================

results_df = run_batch_evaluation(available_recordings)

print("\n" + "=" * 60)
print("PER-RECORDING RESULTS")
print("=" * 60)
display_cols = ['recording_id', 'style', 'is_comp',
                'key_score', 'chord_majmin_acc', 'note_f1']
print(results_df[display_cols].to_string(index=False))

print("\n" + "=" * 60)
print("OVERALL AVERAGES")
print("=" * 60)
summary = results_df[['key_score', 'chord_root_acc', 'chord_majmin_acc',
                       'chord_triads_acc', 'note_precision', 'note_recall', 'note_f1']].mean()
for metric, value in summary.items():
    print(f"  {metric:25s}  {value:.3f}")

print("\n" + "=" * 60)
print("BY INPUT TYPE (Comp vs Solo)")
print("=" * 60)
by_type = results_df.groupby('is_comp')[['key_score', 'chord_majmin_acc', 'note_f1']].mean()
by_type.index = ['Solo', 'Comp']
print(by_type.round(3).to_string())

print("\n" + "=" * 60)
print("BY STYLE")
print("=" * 60)
by_style = results_df.groupby('style')[['key_score', 'chord_majmin_acc', 'note_f1']].mean()
print(by_style.round(3).to_string())

[1/360] Processing 00_BN1-129-Eb_comp...
1/1 [==============================] - 0s 67ms/step
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/00_BN1-129-Eb_comp_mic.wav...
[2/360] Processing 00_BN1-129-Eb_solo...
1/1 [==============================] - 0s 48ms/step
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/00_BN1-129-Eb_solo_mic.wav...
[3/360] Processing 00_BN1-147-Gb_comp...
1/1 [==============================] - 0s 92ms/step
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/00_BN1-147-Gb_comp_mic.wav...
[4/360] Processing 00_BN1-147-Gb_solo...
1/1 [==============================] - 0s 45ms/step
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/00_BN1-147-Gb_solo_mic.wav...
[5/360] Processing 00_BN2-131-B_comp...
1/1 [==============================] - 0s 73ms/step
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/00_BN2-131-